## Import and Installs

In [0]:
import sys
import os
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sys.path.append(os.path.abspath(".."))

from src.config.settings import LakehouseConfig
from src.quality.data_contract import (
    validate_contract_clientes,
    validate_contract_transacoes,
)
from src.silver.scd2 import compute_hash_diff, execute_scd2_merge

In [0]:
config = LakehouseConfig()
spark.sql(f"USE CATALOG {config.catalog}")
spark.sql(f"USE SCHEMA {config.schema_silver}")

## load contrato

In [0]:
contract_full_path = (
    f"/Volumes/{config.catalog}/silver/stg/contracts/silver_quality_contract.json"
)

with open(contract_full_path, "r", encoding="utf-8") as f:
    silver_contract = json.load(f)

cli_rules = silver_contract["entities"]["clientes"]
tx_rules = silver_contract["entities"]["transacoes"]
print("Contratos de qualidade carregados com sucesso.")

## quarentena e scd t2

In [0]:
df_bronze_cli = spark.table(
    f"{config.catalog}.{config.schema_bronze}.bronze_clientes_cdc"
)
cli_validos, cli_quarentena = validate_contract_clientes(df_bronze_cli, cli_rules)

# write quarentena
cli_quarentena.write.format("delta").mode("append").saveAsTable(
    f"{config.catalog}.{config.schema_silver}.quarentena_clientes"
)

# apply SCD T2
business_cols = ["cpf", "nome", "cidade", "estado", "renda", "segmento"]
cli_com_hash = compute_hash_diff(cli_validos, business_cols)

execute_scd2_merge(
    spark=spark,
    target_table_name=f"{config.catalog}.{config.schema_silver}.dim_cliente_scd2",
    df_updates=cli_com_hash,
    primary_key="id_cliente",
)

## transacoes (quarentena + dedup + reconsiliacao)

In [0]:
df_bronze_tx = spark.table(f"{config.catalog}.{config.schema_bronze}.bronze_transacoes")
tx_validas, tx_quarentena = validate_contract_transacoes(df_bronze_tx, tx_rules)

tx_quarentena.write.format("delta").mode("append").saveAsTable(
    f"{config.catalog}.{config.schema_silver}.quarentena_transacoes"
)

w_tx = Window.partitionBy("id_transacao").orderBy(F.col("_timestamp_ingestao").desc())
tx_dedup = (
    tx_validas.withColumn("rn", F.row_number().over(w_tx))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_estornos = spark.table(
    f"{config.catalog}.{config.schema_bronze}.bronze_estornos"
).select("id_transacao", "data_estorno", "motivo")

tx_final = (
    tx_dedup.join(df_estornos, on="id_transacao", how="left")
    .withColumn("data_transacao", F.to_timestamp(F.col("data_transacao")))
    .withColumn("flag_estornado", F.col("data_estorno").isNotNull())
    .withColumn(
        "status_transacao",
        F.when(F.col("flag_estornado") == True, F.lit("ESTORNADA")).otherwise(
            F.lit("APROVADA")
        ),
    )
    .withColumn(
        "valor_liquido",
        F.when(
            F.col("flag_estornado") == True, F.lit(0.00).cast("decimal(15,2)")
        ).otherwise(F.col("valor").cast("decimal(15,2)")),
    )
    .withColumn("motivo_estorno", F.col("motivo"))
    .withColumn("data_processamento_prata", F.current_timestamp())
    .drop("motivo")
)

(
    tx_final.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{config.catalog}.{config.schema_silver}.silver_transacoes")
)

In [0]:
print("Processamento da camada silver concluido com sucesso!!")